# Live Trading with CCXT

Learn how to deploy your strategies for live trading using CCXT integration.

## Topics

1. CCXT setup and configuration
2. Connecting to exchanges
3. Live data feeds
4. Order execution
5. Risk management in live trading
6. Monitoring and logging

## ⚠️ Important Warnings

**Before trading with real money:**

1. ✅ Test thoroughly in paper trading mode
2. ✅ Start with small amounts
3. ✅ Understand exchange fees and limits
4. ✅ Implement proper error handling
5. ✅ Monitor your bot continuously
6. ✅ Have stop-loss mechanisms

**Never:**
- ❌ Trade with money you can't afford to lose
- ❌ Leave bots running unmonitored
- ❌ Use API keys with withdrawal permissions
- ❌ Skip paper trading phase

## 1. CCXT Setup

Install and configure CCXT for Backtrader.

In [ ]:
# Install required packages
# !pip install ccxt backtrader

import backtrader as bt
import ccxt
from datetime import datetime
import os

# Check CCXT version
print(f"CCXT version: {ccxt.__version__}")
print(f"Backtrader version: {bt.__version__}")

# List supported exchanges
print(f"\nSupported exchanges: {len(ccxt.exchanges)}")
print(f"Popular exchanges: {', '.join(ccxt.exchanges[:10])}...")

## 2. Exchange Configuration

Set up exchange connection (using testnet/sandbox).

In [ ]:
# Example: Binance Testnet Configuration
# NEVER hardcode real API keys in code!

exchange_config = {
    'apiKey': os.getenv('BINANCE_TESTNET_API_KEY', 'your_testnet_api_key'),
    'secret': os.getenv('BINANCE_TESTNET_SECRET', 'your_testnet_secret'),
    'enableRateLimit': True,
    'options': {
        'defaultType': 'future',  # or 'spot'
        'adjustForTimeDifference': True,
    }
}

# For testnet
exchange_config['urls'] = {
    'api': {
        'public': 'https://testnet.binancefuture.com/fapi/v1',
        'private': 'https://testnet.binancefuture.com/fapi/v1',
    }
}

# Initialize exchange
exchange = ccxt.binance(exchange_config)

# Test connection
try:
    balance = exchange.fetch_balance()
    print("✅ Connected to exchange")
    print(f"Account balance: {balance['total']}")
except Exception as e:
    print(f"❌ Connection failed: {e}")

## 3. Live Data Feed

Create a live data feed using CCXT.

In [ ]:
from backtrader.feeds import ccxtfeed

class LiveStrategy(bt.Strategy):
    """Simple live trading strategy."""
    
    params = (
        ('sma_period', 20),
        ('risk_percent', 0.01),  # Risk 1% per trade
    )
    
    def __init__(self):
        self.sma = bt.indicators.SMA(period=self.p.sma_period)
        self.order = None
    
    def log(self, txt, dt=None):
        """Logging function."""
        dt = dt or self.data.datetime.datetime(0)
        print(f'{dt.isoformat()} {txt}')
    
    def notify_order(self, order):
        if order.status in [order.Submitted, order.Accepted]:
            return
        
        if order.status == order.Completed:
            if order.isbuy():
                self.log(f'BUY EXECUTED, Price: {order.executed.price:.2f}, '
                        f'Size: {order.executed.size:.4f}')
            else:
                self.log(f'SELL EXECUTED, Price: {order.executed.price:.2f}, '
                        f'Size: {order.executed.size:.4f}')
        
        elif order.status in [order.Canceled, order.Margin, order.Rejected]:
            self.log(f'Order {order.status}')
        
        self.order = None
    
    def notify_trade(self, trade):
        if trade.isclosed:
            self.log(f'TRADE PROFIT, Gross: {trade.pnl:.2f}, Net: {trade.pnlcomm:.2f}')
    
    def next(self):
        # Don't trade if order pending
        if self.order:
            return
        
        # Log current state
        self.log(f'Close: {self.data.close[0]:.2f}, SMA: {self.sma[0]:.2f}')
        
        # Trading logic
        if not self.position:
            if self.data.close[0] > self.sma[0]:
                # Calculate position size based on risk
                cash = self.broker.getcash()
                size = (cash * self.p.risk_percent) / self.data.close[0]
                self.log(f'BUY CREATE, Size: {size:.4f}')
                self.order = self.buy(size=size)
        else:
            if self.data.close[0] < self.sma[0]:
                self.log('SELL CREATE')
                self.order = self.close()

# Note: Actual live trading setup
# cerebro = bt.Cerebro()
# 
# # Add CCXT store
# from backtrader.stores import ccxtstore
# store = ccxtstore.CCXTStore(
#     exchange='binance',
#     currency='USDT',
#     config=exchange_config,
#     retries=5,
# )
# 
# # Add live data feed
# data = store.getdata(
#     dataname='BTC/USDT',
#     timeframe=bt.TimeFrame.Minutes,
#     compression=5,
#     ohlcv_limit=100,
# )
# cerebro.adddata(data)
# 
# # Add broker
# broker = store.getbroker()
# cerebro.setbroker(broker)
# 
# # Add strategy
# cerebro.addstrategy(LiveStrategy)
# 
# # Run live
# cerebro.run()

print("Live trading setup example (commented out for safety)")

## 4. Paper Trading Mode

Test your strategy without risking real money.

In [ ]:
# Paper trading configuration
paper_config = exchange_config.copy()
paper_config['options']['defaultType'] = 'future'
paper_config['sandbox'] = True  # Enable sandbox mode

# Initialize paper trading exchange
paper_exchange = ccxt.binance(paper_config)

# Test paper trading
try:
    # Fetch ticker
    ticker = paper_exchange.fetch_ticker('BTC/USDT')
    print(f"BTC/USDT Price: ${ticker['last']:.2f}")
    
    # Fetch balance
    balance = paper_exchange.fetch_balance()
    print(f"Paper account balance: {balance['USDT']['free']:.2f} USDT")
    
except Exception as e:
    print(f"Paper trading test failed: {e}")

## 5. Risk Management

Essential risk controls for live trading.

In [ ]:
class SafeLiveStrategy(bt.Strategy):
    """Live strategy with comprehensive risk management."""
    
    params = (
        ('max_position_size', 0.1),    # Max 10% of portfolio per position
        ('max_daily_loss', 0.02),      # Stop trading if lose 2% in a day
        ('max_total_exposure', 0.5),   # Max 50% of portfolio in positions
        ('stop_loss_pct', 0.02),       # 2% stop-loss
        ('take_profit_pct', 0.05),     # 5% take-profit
    )
    
    def __init__(self):
        self.sma = bt.indicators.SMA(period=20)
        self.atr = bt.indicators.ATR(period=14)
        
        self.daily_start_value = self.broker.getvalue()
        self.daily_pnl = 0
        self.trading_enabled = True
        
        self.orders = {}
    
    def check_daily_loss(self):
        """Check if daily loss limit exceeded."""
        current_value = self.broker.getvalue()
        daily_loss = (self.daily_start_value - current_value) / self.daily_start_value
        
        if daily_loss > self.p.max_daily_loss:
            self.log(f'⚠️ DAILY LOSS LIMIT EXCEEDED: {daily_loss:.2%}')
            self.trading_enabled = False
            # Close all positions
            if self.position:
                self.close()
    
    def check_exposure(self):
        """Check total portfolio exposure."""
        total_value = self.broker.getvalue()
        position_value = abs(self.position.size * self.data.close[0])
        exposure = position_value / total_value
        
        return exposure < self.p.max_total_exposure
    
    def calculate_position_size(self):
        """Calculate safe position size."""
        total_value = self.broker.getvalue()
        max_position_value = total_value * self.p.max_position_size
        
        # ATR-based sizing
        risk_amount = total_value * 0.01  # Risk 1%
        atr_stop = self.atr[0] * 2
        size = risk_amount / atr_stop if atr_stop > 0 else 0
        
        # Cap at max position size
        max_size = max_position_value / self.data.close[0]
        return min(size, max_size)
    
    def next(self):
        # Check daily loss limit
        self.check_daily_loss()
        
        if not self.trading_enabled:
            return
        
        if not self.position:
            # Entry conditions
            if (self.data.close[0] > self.sma[0] and 
                self.check_exposure()):
                
                size = self.calculate_position_size()
                if size > 0:
                    # Entry order
                    entry_order = self.buy(size=size)
                    
                    # Calculate stop and target
                    entry_price = self.data.close[0]
                    stop_price = entry_price * (1 - self.p.stop_loss_pct)
                    target_price = entry_price * (1 + self.p.take_profit_pct)
                    
                    # Store for bracket orders
                    self.orders['entry'] = entry_order
                    self.log(f'Entry: {entry_price:.2f}, '
                            f'Stop: {stop_price:.2f}, '
                            f'Target: {target_price:.2f}')
        else:
            # Exit conditions
            if self.data.close[0] < self.sma[0]:
                self.close()
    
    def log(self, txt, dt=None):
        dt = dt or self.data.datetime.datetime(0)
        print(f'{dt.isoformat()} {txt}')

print("Safe live strategy defined with:")
print("- Daily loss limit")
print("- Position size limits")
print("- Total exposure control")
print("- ATR-based stops")

## 6. Monitoring and Logging

Set up proper monitoring for live trading.

In [ ]:
import logging
from datetime import datetime

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(f'live_trading_{datetime.now():%Y%m%d}.log'),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger('LiveTrading')

class MonitoredStrategy(bt.Strategy):
    """Strategy with comprehensive logging."""
    
    def __init__(self):
        self.sma = bt.indicators.SMA(period=20)
        logger.info('Strategy initialized')
    
    def next(self):
        # Log every bar
        logger.debug(f'Bar: Close={self.data.close[0]:.2f}, '
                    f'SMA={self.sma[0]:.2f}')
        
        # Your trading logic here
        pass
    
    def notify_order(self, order):
        if order.status == order.Completed:
            logger.info(f'Order completed: {order.info}')
        elif order.status in [order.Canceled, order.Rejected]:
            logger.warning(f'Order {order.status}: {order.info}')
    
    def notify_trade(self, trade):
        if trade.isclosed:
            logger.info(f'Trade closed: PnL={trade.pnlcomm:.2f}')
    
    def stop(self):
        logger.info(f'Strategy stopped. Final value: {self.broker.getvalue():.2f}')

print("Logging configured. Logs will be saved to file.")

## 7. Deployment Checklist

Before going live:

### Pre-Deployment
- [ ] Tested in paper trading for at least 1 week
- [ ] Verified all API credentials
- [ ] Configured proper logging
- [ ] Set up monitoring alerts
- [ ] Implemented stop-loss mechanisms
- [ ] Tested error handling
- [ ] Reviewed exchange fees and limits

### Risk Controls
- [ ] Daily loss limit configured
- [ ] Position size limits set
- [ ] Total exposure limit set
- [ ] Emergency stop mechanism ready
- [ ] Backup plan prepared

### Monitoring
- [ ] Real-time monitoring dashboard
- [ ] Alert system configured
- [ ] Log rotation set up
- [ ] Performance metrics tracked
- [ ] Error notification system

### Post-Deployment
- [ ] Monitor first 24 hours closely
- [ ] Review logs daily
- [ ] Track performance metrics
- [ ] Adjust parameters as needed
- [ ] Document all issues

## Summary

You've learned:
- ✅ CCXT setup and configuration
- ✅ Exchange connection
- ✅ Live data feeds
- ✅ Paper trading
- ✅ Risk management
- ✅ Monitoring and logging

## Critical Reminders

1. **Start small** - Begin with minimal capital
2. **Paper trade first** - Test for weeks before going live
3. **Monitor constantly** - Never leave bots unattended
4. **Use stop-losses** - Always protect your capital
5. **Log everything** - Track all trades and errors
6. **Stay updated** - Monitor exchange announcements

## Resources

- [CCXT Documentation](https://docs.ccxt.com/)
- [Backtrader CCXT Guide](../live-trading/ccxt-guide.md)
- [Risk Management Guide](/advanced/risk-management.md)
- [Exchange API Docs](https://binance-docs.github.io/apidocs/)

## Disclaimer

Trading cryptocurrencies carries significant risk. This tutorial is for educational purposes only. Always:
- Do your own research
- Understand the risks
- Never invest more than you can afford to lose
- Consult with financial advisors